In [1]:
import os
import cv2
import sys
import time
import joblib
import numpy as np

from sklearn.metrics import accuracy_score, classification_report
from skimage.filters import gabor
from skimage.feature import local_binary_pattern
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

IMG_SIZE = 128
frequencies = [0.1, 0.2, 0.3, 0.4]
thetas = [0, np.pi/4, np.pi/2, 3*np.pi/4]


def load_image(path):
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))
    color = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return gray, color


def gabor_features(image):
    features = []
    for theta in thetas:
        for freq in frequencies:
            real, _ = gabor(image, frequency=freq, theta=theta)
            features.extend([real.mean(), real.var(), real.std(), np.max(real)])
    return features


def lbp_features(image):
    lbp = local_binary_pattern(image, P=8, R=1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10))
    return hist.tolist()


def color_features(image):
    hist = cv2.calcHist([image], [0,1,2], None, [8,8,8],[0,256,0,256,0,256])
    return hist.flatten().tolist()


def extract_features(gray, color):
    return gabor_features(gray) + lbp_features(gray) + color_features(color)


def load_dataset(path):
    X, y = [], []

    for label in os.listdir(path):
        class_path = os.path.join(path, label)
        if not os.path.isdir(class_path):
            continue

        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            try:
                gray, color = load_image(img_path)
                features = extract_features(gray, color)
                X.append(features)
                y.append(label)
            except:
                continue

    return np.array(X), np.array(y)


if __name__ == "__main__":

    BASE = "/kaggle/input/datasets/medasivamanoj/leaf-disease"

    train_path = f"{BASE}/train1/train1"
    val_path   = f"{BASE}/validation1/validation1"
    test_path  = f"{BASE}/test1/test1"

    OUT_DIR = "/kaggle/working"
    

    print("Extracting features...")
    X_train, y_train = load_dataset(train_path)
    X_val, y_val = load_dataset(val_path)
    X_test, y_test = load_dataset(test_path)

    np.save(f"{OUT_DIR}/X_train.npy", X_train)
    np.save(f"{OUT_DIR}/y_train.npy", y_train)

    np.save(f"{OUT_DIR}/X_val.npy", X_val)
    np.save(f"{OUT_DIR}/y_val.npy", y_val)

    np.save(f"{OUT_DIR}/X_test.npy", X_test)
    np.save(f"{OUT_DIR}/y_test.npy", y_test)

    print("Features saved!")

    print("Creating scaler...")
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    joblib.dump(scaler, f"{OUT_DIR}/scaler.pkl")

    
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        class_weight='balanced'
        )
    
    rf.fit(X_train, y_train)

    joblib.dump(rf, f"{OUT_DIR}/random_forest_model.pkl")
    
    print("Model saved successfully!")

    models = {
        "Random Forest": rf,
    }

   

    print("\n ======== VALIDATION RESULTS =======")
    for name, model in models.items():
        y_pred = model.predict(X_val)
        print(f"\n{name}")
        print("Accuracy:", accuracy_score(y_val, y_pred))
        print(classification_report(y_val, y_pred))


    print("\n===== TEST RESULTS =====")
    for name, model in models.items():
        y_pred = model.predict(X_test)
        print(f"\n{name}")
        print("Accuracy:", accuracy_score(y_test, y_pred))
        print(classification_report(y_test, y_pred))



Extracting features...
Features saved!
Creating scaler...
Model saved successfully!

 ======== VALIDATION RESULTS =======

Random Forest
Accuracy: 0.8174342105263158
                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.70      0.44      0.54        16
                                 Apple___Black_rot       0.94      0.94      0.94        16
                          Apple___Cedar_apple_rust       0.68      0.94      0.79        16
                                   Apple___healthy       0.68      0.81      0.74        16
                               Blueberry___healthy       0.82      0.88      0.85        16
          Cherry_(including_sour)___Powdery_mildew       0.82      0.88      0.85        16
                 Cherry_(including_sour)___healthy       0.94      0.94      0.94        16
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.88      0.94      0.91        16
     